# TensorRT engine build in a hosted environment

Install TensorRT-LLM and HuggingFace prerequisites

In [ ]:
%pip install tensorrt_llm -U --pre --extra-index-url https://pypi.nvidia.com --quiet
%pip install huggingface_hub mpi4py --quiet

Get TensorRT-LLM version number and checkout the respective branch

In [ ]:
import os

os.environ['TRT_LLM_VERSION'] = os.popen("pip show tensorrt-llm | sed -n 's/^Version: //p'").read().strip()

!git clone --branch "v${TRT_LLM_VERSION}" https://github.com/NVIDIA/TensorRT-LLM.git
%cd TensorRT-LLM/examples/models/core/llama

Fix CUDA toolkit mismatches (needed in Google Colab and other hosted envs)

In [ ]:
! apt -q update
! apt -q install -y libcublas-13-0

NVRTC_PATH = "/usr/local/lib/python3.12/dist-packages/nvidia/cu13/lib"
if NVRTC_PATH not in os.environ.get('LD_LIBRARY_PATH', ''):
    os.environ['LD_LIBRARY_PATH'] = f"{NVRTC_PATH}:{os.environ.get('LD_LIBRARY_PATH', '')}"

Download model weights

In [ ]:
repo_id="NousResearch/Llama-2-7b-hf"  # small enough for hosted envs

from huggingface_hub import snapshot_download, notebook_login
notebook_login()
model_dir = snapshot_download(repo_id=repo_id)

Convert Hugging Face checkpoints to Tensor RT-LLM format

(fix manually for Llama-2-7b-hf)

In [ ]:
# Assuming model_dir is the path where the HF model was downloaded
OUTPUT_DIR="./tmp/trt_checkpoints/llama-2-7b"
! python convert_checkpoint.py --model_dir $model_dir \
                               --output_dir $OUTPUT_DIR \
                               --dtype float16

Build the TensorRT Engine

In [ ]:
ENGINE_DIR="./tmp/trt_engines/llama-2-7b"
! trtllm-build --checkpoint_dir $OUTPUT_DIR \
               --output_dir $ENGINE_DIR \
               --gpt_attention_plugin float16 \
               --gemm_plugin float16 \
               --max_input_len 2048 \
               --max_output_len 2048 \
               # Other parameters like --max_batch_size, --tp_size (tensor-parallelism)